# Ⅱ-07 · 딥러닝으로 독버섯 가려내기 — **정답용**

교과서 **126~130쪽**. 빈칸이 모두 채워져 있는 완성본입니다.

> 진도가 빠른 학생, 결석해서 따라가야 하는 학생, 그리고 화면에 띄워 함께 볼 때 씁니다.
> 처음 해 보는 것이라면 **학생용**을 먼저 열어 직접 채워 보세요.

---

1. 맨 위 **파일 → 드라이브에 사본 저장**.
2. 셀 왼쪽 **▶** 또는 **Shift + Enter**.
3. **위에서부터 차례로** 실행하세요.


## 오늘 하는 일

지난 시간 모델과 **순서는 똑같습니다.** ① 읽고 ② 나누고 ③ 학습시키고 ④ 점수.
달라지는 것은 **③번 «모델» 자리**뿐입니다 — 거기에 **층을 쌓습니다.**

| 지난 시간 | 오늘 |
|---|---|
| `model = KNeighborsClassifier()` | `model = keras.Sequential()` + 층 세 줄 |
| 사람이 **어떤 속성을 볼지 골라 줌** | 모델이 **무엇을 볼지 스스로 찾음** |
| 금방 끝남 | **시간이 오래 걸림** |

> ### ⚠ 먼저 읽어 주세요
> 이 노트북은 **독버섯 판별 모델**을 만듭니다. **연습입니다.**
> 여기서 만든 것으로 **실제 버섯을 먹을지 정하면 절대 안 됩니다.**
> 왜 그런지는 맨 아래에서 이야기합니다.

---
## ① 데이터 읽기

버섯 **8,124개**의 생김새가 들어 있습니다.
`class` 열이 정답입니다 — **e**(edible, 먹을 수 있음) / **p**(poisonous, 독버섯).

In [ ]:
import pandas as pd
from tensorflow import keras
import matplotlib.pyplot as plt

주소 = 'https://raw.githubusercontent.com/richee-pc/AI_cs/main/data/'
df = pd.read_csv(주소 + 'mushrooms.csv')

print(df.shape)
df.head()

`(8124, 23)` — 8,124줄에 23열입니다.

정답이 몇 대 몇인지 봅시다. **한쪽으로 치우쳐 있으면 학습이 잘 안 됩니다.**

In [ ]:
df['class'].value_counts()

먹을 수 있는 것 **4,208**, 독버섯 **3,916**. 거의 반반이라 좋습니다.

> 만약 **독버섯이 1%** 뿐이었다면?
> «전부 먹을 수 있다»고만 답해도 정확도 99% 가 나옵니다. 그런데 그 모델은 쓸모가 없지요.
> 이런 것을 **데이터가 치우쳤다**고 합니다. Ⅱ-01 에서 배운 **편향**입니다.

---
## ② 글자를 숫자로 바꾸기

`df.head()` 를 보면 값이 전부 **글자**입니다 — `x`, `s`, `n`, `t`…
**모델은 숫자만 먹습니다.** 그래서 글자를 숫자로 바꿔 줍니다.

`LabelEncoder` 가 그 일을 합니다. `e`→0, `p`→1 처럼 **종류마다 번호**를 붙입니다.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for 열이름 in df.columns:
    df[열이름] = le.fit_transform(df[열이름])

df.head()

전부 숫자가 되었지요?

> `for` 가 나왔습니다. **«모든 열에 대해 같은 일을 되풀이하라»** 는 뜻입니다.
> 23번 손으로 쓰는 대신 세 줄로 끝냈습니다.

---
## ③ 속성(x)과 정답(y) 나누기

`class` 는 **맨 앞(0번째) 열**입니다. 그러니 **0번째가 정답 y**, **1번째부터 끝까지가 속성 x**.

`iloc` 은 «이름 말고 **번호**로 고르겠다»는 뜻입니다.
`[:, 1:]` 은 «**모든 줄**, 그리고 **1번째 열부터 끝까지**» 입니다.

> **힌트** · 정답은 맨 앞 열 하나입니다. 번호 한 자리만 적으면 됩니다.

In [ ]:
x = df.iloc[:, 1:]
y = df.iloc[:, 0]

print('속성 x :', x.shape)
print('정답 y :', y.shape)

x 는 `(8124, 22)`, y 는 `(8124,)` 입니다.

---
## ④ 훈련용과 시험지 나누기 — 지난 시간과 똑같습니다

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=10)

x_train.shape, x_test.shape

---
## ⑤ 층 쌓기 — 오늘의 핵심

`Sequential()` 은 **빈 모델**입니다. 여기에 층을 **차곡차곡 쌓습니다.**

`Dense(64, ...)` 는 «**퍼셉트론 64개**가 들어 있는 층»이라는 뜻입니다.
[수업 자료에서 눌러 본 그 퍼셉트론](https://richee-pc.github.io/AI_cs/unit2.html#nn)이 64개 들어 있다고 보면 됩니다.

**활성화 함수**도 정해 줍니다.
- **relu** — 0보다 작으면 0, 크면 그대로. 은닉층에 흔히 씁니다
- **sigmoid** — 0~1 사이 **확률**로 바꿉니다. «둘 중 하나» 문제의 출력층에 씁니다

> **힌트 ①** · 은닉층 활성화 함수는 위 설명의 첫 번째 것입니다.
> **힌트 ②** · 답이 «먹을 수 있다/없다» **둘 중 하나**라면, 출력층 뉴런은 몇 개면 될까요?

In [ ]:
model = keras.Sequential()

model.add(keras.layers.Dense(64, activation='relu'))     # 은닉층 ①
model.add(keras.layers.Dense(8,  activation='relu'))     # 은닉층 ②
model.add(keras.layers.Dense(1, activation='sigmoid'))   # 출력층

세 줄로 신경망을 만들었습니다. 그림으로 그리면 이렇게 생겼습니다.

```
입력 22개  →  은닉층 64개  →  은닉층 8개  →  출력 1개
              (relu)         (relu)       (sigmoid)
```

**은닉층이 둘**이니 «깊다»고 할 수 있고, 그래서 **딥**러닝입니다.
2016년 이세돌 9단과 둔 알파고는 은닉층이 **13개**였습니다.

---
## ⑥ 학습 방법 정하기 (compile)

- `optimizer='adam'` — **경사하강법을 똑똑하게** 하는 방법. 방향과 보폭을 알아서 조절합니다
- `loss=...` — **얼마나 틀렸는지 재는 자**(손실함수)
- `metrics=['accuracy']` — 학습 중에 **정확도도 같이 보여 달라**

손실함수는 **무엇을 맞히느냐**에 따라 다릅니다.

| 맞히는 것 | 손실함수 |
|---|---|
| 숫자 | `mean_squared_error` |
| **둘 중 하나** | `binary_crossentropy` |
| 셋 이상 중 하나 | `categorical_crossentropy` |

> **힌트** · 우리 문제는 «먹을 수 있다 / 없다» 둘 중 하나입니다. 위 표에서 고르세요.

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

---
## ⑦ 학습시키기

- `epochs` — 전체 데이터를 **몇 번 되풀이해 볼지**
- `batch_size` — **몇 개마다 한 번씩** 가중치를 고칠지

`epochs=15`, `batch_size=256` 으로 해 봅시다.

> **힌트** · 열다섯 번 되풀이합니다.

In [ ]:
history = model.fit(x_train, y_train, epochs=15, batch_size=256)

줄이 열다섯 개 주르륵 나왔지요? **한 줄이 1 에포크**입니다.

`loss` 가 **점점 줄고**, `accuracy` 가 **점점 오르는** 것을 보세요.
**틀린 만큼 되짚어 가며 가중치를 고치는 일(오차 역전파)** 이 눈앞에서 일어난 겁니다.

그림으로 보면 더 뚜렷합니다.

In [ ]:
plt.plot(history.history['loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.show()

오른쪽으로 갈수록 **뚝 떨어지는** 모양이면 학습이 잘된 것입니다.

---
## ⑧ 시험지로 점수 내기

학습에 **안 쓴** 데이터로 재야 진짜 실력입니다.

> **힌트** · 점수를 매기는 명령은 «평가하다」라는 영어 낱말입니다.

In [ ]:
model.evaluate(x_test, y_test)

`[손실함수값, 정확도]` 순서로 나옵니다. 교과서 129쪽은 **정확도 약 0.975** 입니다.

### 과적합이 아닌지 확인하기

**학습할 때의 정확도**와 **시험지 정확도**를 견줍니다.
학습 때는 높은데 시험지에서 뚝 떨어지면 → **과적합**(연습 문제만 외운 상태)입니다.

In [ ]:
print('학습할 때 정확도 :', history.history['accuracy'][-1])
print('시험지 정확도    :', model.evaluate(x_test, y_test, verbose=0)[1])

두 숫자가 **비슷하면** 과적합이 아닙니다. 다행이네요.

---
## ⑨ 한 줄로 성능 올리기 — 정규화

속성마다 **값의 범위가 제각각**이면, 큰 숫자를 가진 속성 쪽으로 모델이 쏠립니다.
그래서 전부 **0~1 사이로 맞춰** 줍니다. 이것을 **정규화**라고 합니다.

> [수업 자료의 선형 회귀에서 «50걸음을 눌러도 절편이 느릿느릿»](https://richee-pc.github.io/AI_cs/unit2.html#algo) 하던 것 기억나나요?
> 그것이 바로 범위가 안 맞아서 생긴 일이었습니다.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
x2 = scaler.fit_transform(x)

x2[0]     # 첫 줄을 보면 전부 0~1 사이

이 데이터로 **똑같은 모델을 처음부터** 다시 만들어 봅시다.

In [ ]:
x_train2, x_test2, y_train2, y_test2 = train_test_split(x2, y, test_size=0.2, random_state=10)

model2 = keras.Sequential()
model2.add(keras.layers.Dense(64, activation='relu'))
model2.add(keras.layers.Dense(8,  activation='relu'))
model2.add(keras.layers.Dense(1,  activation='sigmoid'))
model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model2.fit(x_train2, y_train2, epochs=15, batch_size=256, verbose=0)

print('정규화 전 :', model.evaluate(x_test,  y_test,  verbose=0))
print('정규화 후 :', model2.evaluate(x_test2, y_test2, verbose=0))

정확도가 올라갔나요? 교과서 130쪽에서는 **0.975 → 0.991** 이 되었습니다.

**코드 한 줄 더 넣은 것뿐인데** 성능이 올랐습니다.
Ⅱ단원 내내 나온 그 말 — **«데이터를 어떻게 다듬느냐가 성능을 가른다»** 가 여기서도 맞았습니다.

> 딥러닝은 **돌릴 때마다 결과가 조금씩 다릅니다.** 처음 가중치를 무작위로 잡기 때문입니다.
> 정확도가 0.98 이 나오든 0.99 가 나오든 놀라지 마세요.

---
## ⚠ 마지막으로 — 이 모델을 믿으면 안 되는 이유

정확도 99%. 훌륭해 보이지요? 그런데 **실제로 쓰면 사람이 죽을 수 있습니다.** 왜일까요?

1. **데이터에 없는 버섯은 못 맞힙니다.**
   이 8,124개는 북미 버섯 몇 종의 기록입니다. 우리나라 산에 나는 버섯은 **들어 있지 않습니다.**

2. **«냄새»처럼 사람이 재기 어려운 속성에 기대고 있습니다.**
   `odor` 열이 정답과 거의 맞아떨어집니다. 산에서 냄새를 정확히 구분할 수 있나요?

3. **1%가 어떤 1%인지가 중요합니다.**
   100개 중 하나를 틀리는데, 그 하나가 **«독버섯인데 먹어도 된다»** 라면
   정확도 99%는 아무 의미가 없습니다. → **재현율**을 봐야 하는 이유입니다.

> **«정확도가 높다»와 «믿고 써도 된다»는 다른 말입니다.**
> 이 질문이 Ⅲ단원 「인공지능의 사회적 영향」과 수행평가 ① 토론으로 이어집니다.

---
## 다 했습니다 · 확인해 보세요

- [ ] `Sequential()` 에 `Dense()` 를 쌓아 신경망을 만든다
- [ ] 은닉층은 **relu**, 둘 중 하나를 고르는 출력층은 **sigmoid** 뉴런 **1개**
- [ ] 둘 중 하나 문제의 손실함수는 **binary_crossentropy**
- [ ] **에포크**는 되풀이 횟수, **배치 사이즈**는 몇 개마다 고칠지
- [ ] 학습 정확도와 시험지 정확도를 견주면 **과적합**을 알 수 있다
- [ ] 정확도가 높아도 **믿고 쓸 수 있는지는 따로 따져야** 한다

### 한 걸음 더 (시간이 남으면)

- `epochs` 를 **50** 으로 올려 보세요. 계속 좋아지나요? 어디서 멈추나요?
- 은닉층을 **하나 더** 쌓아 보세요. 깊을수록 좋을까요?
- `Dense(64)` 를 `Dense(4)` 로 줄여 보세요. 얼마나 나빠지나요?

**정답이 없는 물음들입니다.** 이것을 이리저리 바꿔 보는 일을 **하이퍼파라미터 튜닝**이라고 하고,
잘하는 사람이 좋은 모델을 만듭니다.

막히면 👉 [에러 응급처치 사전](https://richee-pc.github.io/AI_cs/colab.html)

*made by ptp 🐰*